In [1]:
# Install wandb for experiment tracking, transformers for AST, timm for ViT.
# Run this cell FIRST — Kaggle kernels may not have all of these by default.

import subprocess
subprocess.run(["pip", "install", "wandb", "transformers", "timm", "scikit-image", "-q"])

CompletedProcess(args=['pip', 'install', 'wandb', 'transformers', 'timm', 'scikit-image', '-q'], returncode=0)

In [2]:
!pip install transformers librosa soundfile -q

In [3]:
import os, glob, random, numpy as np, pandas as pd
import librosa, torch, soundfile as sf
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']
GENRE2ID = {g:i for i,g in enumerate(GENRES)}
ID2GENRE = {i:g for g,i in GENRE2ID.items()}

ROOT        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{ROOT}/genres_stems'
ESC50_DIR   = f'{ROOT}/ESC-50-master/audio'
TEST_CSV    = f'{ROOT}/test.csv'

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SR       = 16000
DURATION = 10
BATCH    = 8
EPOCHS   = 20
LR       = 2e-5
SEED     = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Device:', DEVICE)

2026-03-30 10:22:47.827312: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774866167.985783      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774866168.030992      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774866168.402719      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774866168.402749      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774866168.402752      55 computation_placer.cc:177] computation placer alr

Device: cuda


In [4]:
# ── Build stem index: genre -> list of song dirs ──────────────────────────────
stem_index = {genre: [] for genre in GENRES}
for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    for d in sorted(glob.glob(os.path.join(genre_dir, '*'))):
        if os.path.isdir(d):
            stem_index[genre].append(d)

for g, songs in stem_index.items():
    print(f'{g}: {len(songs)} songs')

# ── Build ESC-50 noise file list ──────────────────────────────────────────────
esc50_files = glob.glob(os.path.join(ESC50_DIR, '*.wav'))
print(f'\nESC-50 noise files: {len(esc50_files)}')

blues: 100 songs
classical: 100 songs
country: 100 songs
disco: 100 songs
hiphop: 100 songs
jazz: 100 songs
metal: 100 songs
pop: 100 songs
reggae: 100 songs
rock: 100 songs

ESC-50 noise files: 2000


In [6]:
def load_stem(song_dir, stem_name, sr=SR, duration=DURATION):
    path = os.path.join(song_dir, stem_name)
    if not os.path.exists(path):
        return np.zeros(sr * duration, dtype=np.float32)
    y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    return y.astype(np.float32)

def pad_or_trim(audio, target_len):
    if len(audio) < target_len:
        return np.pad(audio, (0, target_len - len(audio)))
    return audio[:target_len]

def cross_song_mix(genre, sr=SR, duration=DURATION):
    """
    Core augmentation: pick 4 DIFFERENT songs from the same genre,
    take one stem from each — exactly what the test mashups do.
    """
    songs = stem_index[genre]
    stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
    target_len = sr * duration

    # Sample 4 songs (with replacement if < 4 songs)
    chosen = random.choices(songs, k=4)
    mixed = np.zeros(target_len, dtype=np.float32)
    for song_dir, stem_name in zip(chosen, stem_names):
        stem = pad_or_trim(load_stem(song_dir, stem_name, sr, duration), target_len)
        mixed += stem

    mixed = mixed / (np.abs(mixed).max() + 1e-8)
    return mixed

def same_song_mix(song_dir, sr=SR, duration=DURATION):
    """Fallback: mix all stems from the same song."""
    target_len = sr * duration
    mixed = np.zeros(target_len, dtype=np.float32)
    for stem_name in ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']:
        stem = pad_or_trim(load_stem(song_dir, stem_name, sr, duration), target_len)
        mixed += stem
    mixed = mixed / (np.abs(mixed).max() + 1e-8)
    return mixed

def add_esc50_noise(audio, esc50_files, sr=SR, snr_db=None):
    """Add a random ESC-50 noise clip at random SNR between 10-30 dB."""
    if not esc50_files:
        return audio
    noise_path = random.choice(esc50_files)
    try:
        noise, _ = librosa.load(noise_path, sr=sr, mono=True)
        target_len = len(audio)
        noise = pad_or_trim(noise, target_len)

        # Random SNR between 10 and 30 dB
        snr = snr_db if snr_db else random.uniform(10, 30)
        signal_power = np.mean(audio ** 2) + 1e-8
        noise_power  = np.mean(noise ** 2) + 1e-8
        scale = np.sqrt(signal_power / (noise_power * (10 ** (snr / 10))))
        noisy = audio + scale * noise
        return noisy / (np.abs(noisy).max() + 1e-8)
    except:
        return audio

print('Audio utility functions defined.')

Audio utility functions defined.


In [7]:
# Build flat records list for same-song mixing (used for val only)
records = []
for genre in GENRES:
    for song_dir in stem_index[genre]:
        records.append({'path': song_dir, 'label': GENRE2ID[genre], 'genre': genre})

random.shuffle(records)
split = int(0.85 * len(records))
train_records = records[:split]
val_records   = records[split:]
print(f'Total: {len(records)} | Train: {len(train_records)} | Val: {len(val_records)}')

Total: 1000 | Train: 850 | Val: 150


In [12]:
feature_extractor = ASTFeatureExtractor.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593'
)

class MashupDataset(Dataset):
    def __init__(self, records, augment=False, cross_mix_prob=0.8):
        """
        augment=True  → used for training
        cross_mix_prob → probability of using cross-song mixing vs same-song
        """
        self.records = records
        self.augment = augment
        self.cross_mix_prob = cross_mix_prob
        self.target_len = SR * DURATION

    def __len__(self):
        # During training, oversample to get more augmented variety
        return len(self.records) * (3 if self.augment else 1)

    def __getitem__(self, idx):
        rec = self.records[idx % len(self.records)]
        genre = rec['genre']
        label = rec['label']

        if self.augment and random.random() < self.cross_mix_prob:
            # Cross-song mix: replicates test distribution
            audio = cross_song_mix(genre)
        else:
            # Same-song mix
            audio = same_song_mix(rec['path'])

        if self.augment:
            # Add ESC-50 noise 70% of the time
            if random.random() < 0.7:
                audio = add_esc50_noise(audio, esc50_files)
            # Random gain
            audio = audio * random.uniform(0.8, 1.2)
            # Tiny gaussian noise
            audio = audio + np.random.randn(len(audio)).astype(np.float32) * 0.003

        audio = pad_or_trim(audio.astype(np.float32), self.target_len)
        inputs = feature_extractor(audio, sampling_rate=SR, return_tensors='pt')
        return inputs['input_values'].squeeze(0), label

train_ds = MashupDataset(train_records, augment=True,  cross_mix_prob=0.8)
val_ds   = MashupDataset(val_records,   augment=False, cross_mix_prob=0.0)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 319 | Val batches: 19


In [11]:
model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=10,
    ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = total_steps // 10
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = torch.nn.CrossEntropyLoss()
print(f'Model ready. Total steps: {total_steps} | Warmup: {warmup_steps}')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model ready. Total steps: 6380 | Warmup: 638


In [13]:
best_f1    = 0.0
history    = []

for epoch in range(EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(input_values=inputs).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [val]'):
            preds = model(input_values=inputs.to(DEVICE)).logits.argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    f1      = f1_score(all_labels, all_preds, average='macro')
    avg_loss = total_loss / len(train_loader)
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'f1': f1})
    print(f'Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Val Macro F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        print(f'  >>> Best model saved (F1={f1:.4f})')

print(f'\nTraining complete. Best Val F1: {best_f1:.4f}')
print(pd.DataFrame(history).to_string(index=False))

Epoch 1/20 [val]: 100%|██████████| 19/19 [00:15<00:00,  1.25it/s]


Epoch 01 | Loss: 1.2713 | Val Macro F1: 0.7999
  >>> Best model saved (F1=0.7999)


Epoch 2/20 [val]: 100%|██████████| 19/19 [00:15<00:00,  1.27it/s]


Epoch 02 | Loss: 0.3517 | Val Macro F1: 0.8572
  >>> Best model saved (F1=0.8572)


Epoch 3/20 [val]: 100%|██████████| 19/19 [00:15<00:00,  1.26it/s]


...

Epoch 07 | Loss: 0.0844 | Val Macro F1: 0.9549
  >>> Best model saved (F1=0.9549)


Epoch 8/20 [val]: 100%|██████████| 19/19 [00:15<00:00,  1.27it/s]


Epoch 08 | Loss: 0.0642 | Val Macro F1: 0.9473


Epoch 9/20 [train]:   5%|▌         | 17/319 [00:44<13:06,  2.61s/it]


KeyboardInterrupt: 

In [14]:
# ── Per-class breakdown on validation set ─────────────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/best_model.pt'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc='Final val pass'):
        preds = model(input_values=inputs.to(DEVICE)).logits.argmax(-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=GENRES))

Final val pass: 100%|██████████| 19/19 [00:14<00:00,  1.28it/s]

              precision    recall  f1-score   support

       blues       0.94      1.00      0.97        17
   classical       1.00      1.00      1.00        12
     country       0.95      0.95      0.95        22
       disco       0.94      1.00      0.97        15
      hiphop       0.90      1.00      0.95         9
        jazz       1.00      1.00      1.00        11
       metal       1.00      0.89      0.94        18
         pop       1.00      0.89      0.94        18
      reggae       0.87      1.00      0.93        13
        rock       0.93      0.87      0.90        15

    accuracy                           0.95       150
   macro avg       0.95      0.96      0.95       150
weighted avg       0.96      0.95      0.95       150



In [16]:
def infer_with_tta(audio_path, model, feature_extractor, n_tta=5):
    """
    Test Time Augmentation: run inference n_tta times with slight
    variations and average the logits — boosts score by ~0.02-0.04.
    """
    try:
        audio, _ = librosa.load(audio_path, sr=SR, duration=DURATION, mono=True)
        audio = pad_or_trim(audio.astype(np.float32), SR * DURATION)
    except Exception as e:
        print(f'Load error {audio_path}: {e}')
        return 'rock'

    logits_sum = None
    for i in range(n_tta):
        if i == 0:
            aug = audio.copy()   # first pass: clean
        else:
            # Slight noise + gain variation
            aug = audio + np.random.randn(len(audio)).astype(np.float32) * 0.003
            aug = aug * random.uniform(0.9, 1.1)
            aug = aug / (np.abs(aug).max() + 1e-8)

        inputs = feature_extractor(aug, sampling_rate=SR, return_tensors='pt')
        with torch.no_grad():
            logits = model(input_values=inputs['input_values'].to(DEVICE)).logits
        logits_sum = logits if logits_sum is None else logits_sum + logits

    pred_id = logits_sum.argmax(-1).item()
    return ID2GENRE[pred_id]

print('TTA inference function ready.')

TTA inference function ready.


In [17]:
test_df = pd.read_csv(TEST_CSV)
print(f'Test samples: {len(test_df)}')
print(test_df.head())

predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Inference (TTA)'):
    audio_path = os.path.join(ROOT, row['filename'])
    pred = infer_with_tta(audio_path, model, feature_extractor, n_tta=5)
    predictions.append(pred)

print('Done! Prediction distribution:')
print(pd.Series(predictions).value_counts())

Test samples: 3020
   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav


Inference (TTA): 100%|██████████| 3020/3020 [22:47<00:00,  2.21it/s]

Done! Prediction distribution:
blues        406
hiphop       363
metal        348
disco        346
reggae       331
pop          317
rock         283
jazz         264
country      196
classical    166
Name: count, dtype: int64


In [18]:
sub = pd.DataFrame({'id': test_df['id'], 'genre': predictions})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('submission.csv saved!')
print(sub.head(10))

submission.csv saved!
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7       rock
7   8        pop
8   9        pop
9  10      disco
